# Assignment 4: Phrase Queries and PageRank

**Student names**: Mustafa <br>
**Group number**: 125 <br>
**Date**: 30.10.2026 23:59

In this assignment, we will build indexes for phrase queries, compute PageRank, and answer conceptual questions.

## Dataset and preprocessing

Place `cran.all.1400` next to this notebook. The parser below loads its 1,400 documents into `docs`.
Use only each document's `abstract` field.

In [10]:
# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"cran.all.1400" 

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)

Parsed 1400 documents.


### Preprocessing

Implement `tokenize(text)` and tokenize the abstracts. You may reuse your tokenizer from an earlier assignment, but this time keep the stopwords.

In [11]:
# TODO: Implement preprocessing
import re

def tokenize(text: str) -> list[str]:
    # Convert text to lowercase
    text = text.lower()

    # Extract alphanumeric word tokens, allowing internal hyphens
    tokens = re.findall(r"\b[a-z0-9]+(?:[-'][a-z0-9]+)*\b", text)

    return tokens

# Tokenize the abstracts while keeping stopwords
token_lists = {
    doc_id: tokenize(doc["abstract"])
    for doc_id, doc in docs.items()
}

In [12]:

# Verify that preprocessing worked

print(f"Number of documents parsed: {len(docs)}")
print(f"Number of documents tokenized: {len(token_lists)}")

# Display an example document
doc_id = next(iter(docs))

print(f"\nDocument ID: {doc_id}")
print(f"\nOriginal abstract:\n{docs[doc_id]['abstract']}")
print(f"\nTokens (first 30):\n{token_lists[doc_id][:30]}")

# Check that tokenization produced a list of strings
assert len(docs) == len(token_lists)
assert all(isinstance(token, str)
           for tokens in token_lists.values()
           for token in tokens)

print("\nPreprocessing successful!")


Number of documents parsed: 1400
Number of documents tokenized: 1400

Document ID: 1

Original abstract:
experimental investigation of the aerodynamics of a wing in a slipstream .   an experimental study of a wing in a propeller slipstream was made in order to determine the spanwise distribution of the lift increase due to slipstream at different angles of attack of the wing and at different free stream to slipstream velocity ratios .  the results were intended in part as an evaluation basis for different theoretical treatments of this problem .   the comparative span loading curves, together with supporting evidence, showed that a substantial part of the lift increment produced by the slipstream was due to a /destalling/ or boundary-layer-control effect .  the integrated remaining lift increment, after subtracting this destalling lift, was found to agree well with a potential flow theory .   an empirical evaluation of the destalling effects was made for the specific configuration of t

Converted all abstracts to lowercase and tokenized them into lists of words using regular expressions, Punctuation was removed while internal hyphens and apostrophes were preserved.

## 4.1 Indexes for phrase queries

### 4.1.1 Biword index

A biword is a pair of consecutive tokens. Implement:

- `build_biword_index(token_lists)`: return a dictionary mapping each biword, written as two tokens separated by one space, to sorted, unique document IDs.
- `biword_phrase_search(phrase, index)`: tokenize the phrase and return the sorted IDs of documents containing all its biwords. You may assume at least two query tokens.

In [13]:
# TODO: Build and search the biword index

def build_biword_index(token_lists):
    index = {}

    for doc_id, tokens in token_lists.items():
        for i in range(len(tokens) - 1):
            biword = tokens[i] + " " + tokens[i + 1]

            if biword not in index:
                index[biword] = set()

            index[biword].add(doc_id)

    # Convert document ID sets to sorted lists
    return {
        biword: sorted(doc_ids)
        for biword, doc_ids in index.items()
    }


def biword_phrase_search(phrase, index):
    tokens = tokenize(phrase)

    # Create consecutive pairs from the query
    biwords = [
        tokens[i] + " " + tokens[i + 1]
        for i in range(len(tokens) - 1)
    ]

    if not biwords:
        return []

    # Find documents containing every biword
    result = set(index.get(biwords[0], []))

    for biword in biwords[1:]:
        result.intersection_update(index.get(biword, []))

    return sorted(result)


biword_index = build_biword_index(token_lists)


In [14]:

print("Number of unique biwords:", len(biword_index))

# Inspect an example biword
print("'wing in':", biword_index.get("wing in", [])[:10])

# Search for a phrase
query = "experimental investigation of the aerodynamics"
results = biword_phrase_search(query, biword_index)

print("Query:", query)
print("Number of matching documents:", len(results))
print("Matching document IDs:", results)


Number of unique biwords: 76413
'wing in': [1, 205, 333, 561, 705, 714, 781, 797, 901, 917]
Query: experimental investigation of the aerodynamics
Number of matching documents: 1
Matching document IDs: [1]


Implemented a biword index mapping with consecutive token pairs to sorted, unique document IDs. Implemented phrase search by tokenizing the query and intersecting the document lists for each biword. The index contains 76,413 unique biwords across 1,400 documents. A test phrase correctly returned document 1.

### 4.1.2 Positional index

Implement `build_positional_index(token_lists)` with the structure:

```python
index[term][doc_id] = [position_1, position_2, ...]
```

Store every occurrence, with positions starting at 1 within each document. Keep document IDs and position lists in ascending order.

Then implement `phrase_search(phrase, index)` using the positional index. Return sorted, unique IDs of documents containing the complete phrase.

Return `[]` for an empty query or a query token absent from the index.

In [16]:
# TODO: Build the positional index and implement phrase search

def build_positional_index(token_lists):
    index = {}

    for doc_id, tokens in token_lists.items():
        for position, term in enumerate(tokens, start=1):
            if term not in index:
                index[term] = {}

            if doc_id not in index[term]:
                index[term][doc_id] = []

            index[term][doc_id].append(position)

    return index


def phrase_search(phrase, index):
    tokens = tokenize(phrase)

    if not tokens:
        return []

    # Return no results if any query term is absent
    if any(term not in index for term in tokens):
        return []

    # Candidate documents must contain every query term
    candidates = set(index[tokens[0]])

    for term in tokens[1:]:
        candidates.intersection_update(index[term])

    results = []

    for doc_id in candidates:
        # Positions where the first query term occurs
        first_positions = set(index[tokens[0]][doc_id])

        # Check that each following term occurs consecutively
        for offset, term in enumerate(tokens[1:], start=1):
            positions = set(index[term].get(doc_id, []))
            first_positions = {
                pos for pos in first_positions
                if pos + offset in positions
            }

            if not first_positions:
                break

        if first_positions:
            results.append(doc_id)

    return sorted(results)


positional_index = build_positional_index(token_lists)


In [17]:

print("Number of indexed terms:", len(positional_index))

# Inspect term positions
print("'wing' in document 1:",
      positional_index.get("wing", {}).get(1, []))

# Search for an exact phrase
query = "experimental investigation of the aerodynamics"
results = phrase_search(query, positional_index)

print("Query:", query)
print("Matching document IDs:", results)

# Test empty and missing-term queries
print("Empty query:", phrase_search("", positional_index))
print("Missing term:", phrase_search("xyznonexistent term", positional_index))


Number of indexed terms: 8950
'wing' in document 1: [8, 17, 45]
Query: experimental investigation of the aerodynamics
Matching document IDs: [1]
Empty query: []
Missing term: []


Built a positional index storing each term’s document IDs and occurrence positions, starting at 1. Implemented exact phrase search by checking consecutive term positions. The index contains 8,950 unique terms, and tests confirmed correct phrase matching and handling of empty or missing-term queries.

### 4.1.3 Run phrase queries

Run the following queries with both indexes. The cell prints the number of returned documents and the first five IDs.

In [18]:
phrase_queries = [
    "boundary layer",
    "laminar boundary layer",
    "free stream mach number",
    "in the wind tunnel",
]

for query in phrase_queries:
    biword_docs = biword_phrase_search(query, biword_index)
    phrase_docs = phrase_search(query, positional_index)
    print(f"{query!r}")
    print(f"  Biword:     {len(biword_docs):3d} documents; first five: {biword_docs[:5]}")
    print(f"  Positional: {len(phrase_docs):3d} documents; first five: {phrase_docs[:5]}")

'boundary layer'
  Biword:     291 documents; first five: [2, 3, 4, 7, 8]
  Positional: 291 documents; first five: [2, 3, 4, 7, 8]
'laminar boundary layer'
  Biword:     105 documents; first five: [4, 9, 21, 23, 43]
  Positional:  97 documents; first five: [4, 9, 21, 23, 43]
'free stream mach number'
  Biword:       7 documents; first five: [73, 310, 470, 519, 522]
  Positional:   7 documents; first five: [73, 310, 470, 519, 522]
'in the wind tunnel'
  Biword:       3 documents; first five: [140, 141, 795]
  Positional:   2 documents; first five: [141, 795]


## 4.2 PageRank

The graph in `painters.json` represents 14 Wikipedia pages about painters and 50 directed links between them. An arrow from A to B means that page A links to page B. We will rank the pages within this collection.

![Directed links between the painters. Numbers identify names in the legend.](painters.svg)

The provided cell loads the collection and creates the adjacency matrix `A` as a list of rows. Each row identifies the page a link comes from, and each column identifies the destination page. Thus, `A[i][j] = 1` means that page `i` links to page `j`. If there is no link, the entry is 0.

1. Implement `transition_matrix(A, alpha=0.15)`. Return a new transition matrix `P` with floating-point probabilities, as a list of rows or a NumPy array, where `P[i][j]` is the probability of moving from page `i` to page `j`. Each row must sum to 1.
   - If a page has outgoing links, distribute probability `1 - alpha` equally among them, and probability `alpha` equally among **all** pages, including the current page.
   - If a page has no outgoing links (a dead end), distribute its entire probability equally among all pages.

   For example, Picasso links to Manet and Matisse. With `alpha=0.15`, split the 85% link-following probability between those two destinations, and distribute the remaining 15% across all fourteen pages.

2. Implement `pagerank(P, x0, iterations=50)`. Starting from the provided initial scores `x0`, repeatedly update the score vector using $x_{t+1}=x_tP$ (`x @ P` if you use NumPy arrays). Return the scores after `iterations` updates. This repeated update is called **power iteration**.

In the final cell, the transition matrix you constructed and the `pagerank`-function are used to rank the Wikipedia pages.

In [19]:
import json

with open("painters.json", encoding="utf-8") as f:
    painters = json.load(f)

pages = painters["names"]
n = len(pages)
A = [[0] * n for _ in range(n)]
for source, target in painters["links"]:
    A[source][target] = 1

x0 = [1 / n] * n


In [20]:
# TODO: Construct the transition matrix and compute PageRank
def transition_matrix(A, alpha=0.15):
    n = len(A)
    P = []

    for row in A:
        outlinks = sum(row)

        if outlinks == 0:
            # Dead end: equal probability to every page
            P.append([1.0 / n] * n)
        else:
            # Teleportation probability to every page
            probabilities = [alpha / n] * n

            # Distribute link-following probability
            for j, link in enumerate(row):
                if link:
                    probabilities[j] += (1 - alpha) / outlinks

            P.append(probabilities)

    return P


def pagerank(P, x0, iterations=50):
    x = list(x0)

    for _ in range(iterations):
        x = [
            sum(x[i] * P[i][j] for i in range(len(P)))
            for j in range(len(P))
        ]

    return x


In [21]:
P = transition_matrix(A, alpha=0.15)
scores = pagerank(P, x0)
incoming_links = [sum(A[i][j] for i in range(n)) for j in range(n)]
for i in sorted(range(n), key=lambda i: scores[i], reverse=True):
    print(f"{pages[i]:24} PageRank: {scores[i]:.5f}   Incoming links: {incoming_links[i]}")


Leonardo da Vinci        PageRank: 0.16306   Incoming links: 3
Michel Angelo            PageRank: 0.12954   Incoming links: 4
Peter Paul Rubens        PageRank: 0.11297   Incoming links: 4
Edouard Manet            PageRank: 0.10585   Incoming links: 6
Pierre-Auguste Renoir    PageRank: 0.08147   Incoming links: 5
Claude Monet             PageRank: 0.06720   Incoming links: 5
Pablo Picasso            PageRank: 0.05896   Incoming links: 5
Edgar Degas              PageRank: 0.05144   Incoming links: 3
Paul Cezanne             PageRank: 0.04985   Incoming links: 4
Henri Matisse            PageRank: 0.04966   Incoming links: 3
Rembrandt                PageRank: 0.04814   Incoming links: 2
Vincent van Gogh         PageRank: 0.04459   Incoming links: 4
Gustav Klimt             PageRank: 0.01863   Incoming links: 1
Egon Schiele             PageRank: 0.01863   Incoming links: 1


## 4.3 Conceptual questions

Answer briefly, using your results where relevant.

1. Why can the biword index return extra documents for longer phrases? How does the positional index avoid these matches?
2. Does the painter with the highest PageRank also have the most incoming links? Explain what PageRank captures beyond counting incoming links.
3. Why does PageRank include teleportation?
4. Why might a crawler fetch a lower-priority URL from one host before a higher-priority URL from another host?

*Your answers here*

1. The biword index can return extra documents because it checks whether all consecutive word pairs appear, but not whether they occur in the correct sequence. The positional index verifies that terms occur at consecutive positions in the correct order.

2. Not necessarily. PageRank considers both the number and the importance of incoming links, so links from highly ranked pages contribute more than links from less important pages.

3. Teleportation allows users to jump to any page, prevents pages from becoming unreachable due to dead ends, and helps PageRank converge.

4. Crawlers may enforce per-host limits or politeness policies to avoid overloading servers, so a lower-priority URL from one host may be fetched before a higher-priority URL from another host.